Parameters

In [1]:
## saveto
saveto = "C:\\Users\\Ankith Mohan\\Downloads\\NISQ SDPv2\\"
## saveto = "C:\\Physics\\SDP\\"
# saveto = "/home/kwek123/PythonCode/results/"

## seed
seed = 1

## number of qubits
n_qubits = 6

## ini_type
### state to use as ansatz for k-moment expansion 
### initial state for ansatz state via K moment
### Options:
### 0: + product state
### 1: zero state
### 2: random circuit
### 15: circuit annealing with fixed anneal time
### 16: circuit annealing where annealing time is optimized
ini_type = 2

## annealing time for ini_type = 15
annealtime = 0.3 #0.66

## #(layers) of circuit
depth = 10

## model hamiltonian used
### Options:
### 0: transverse ising
### 1: Heisenberg
### 14: transverse ising model with longitudinal field
### 30: POVM
model = 0

## model = {0, 1, 14}, do SDP or solve generalized eigenvalue problem, 
## model = 30 always uses SDP
do_SDP = True #False

## h parameter for the models
h = 1

## J parameter for the models
J = 1

## g parameter for the models
g = J

## parameters for model = 30 POVM
### slack overlap for POVM 
slack_overlap = 0
### set to values between 0 to 1 to fix overlap between states to be discriminated
### set to -1 to simply choose random states for state discriminiation
fix_overlap = -1
### number of parameters for scanning over overlap between states to be discriminated
n_params = 1
## number of POVMS for model = 30 to discriminate, so far only = 2 implemented
n_POVM = 2

## do conserved number calculation, only works for SDP
get_conserved_number = True # whether to try to get particular conservation number
conserved_number = 0 # get lowest energy eigenstate with conservation of number of particles in SDP

## range of expansion
### how many orders of K-moment expansion should be performed
max_expansion = 2
### maximal number of ansatz states, set to zero to get maximal possible number given max_expansion
max_states = 0

## conditioning factor for inversion and QAE, increase if norm is diverging
inv_cond =  1e-10

levels = 2

# to initialize variable for annealing ansatz
anneal_time_opt = 0

# define which pauli operator one wants to measure of resulting state
target_paulistrings = []
n_target_paulistrings = 0
# target_paulistrings = [np.zeros(n_qubits, dtype = int) for _ in range(n_target_paulistrings)]
# target_paulistrings[0][0] = 1 #0: Identity, 1: X , 2: Y, 3: Z

Libraries

In [2]:
import numpy as np
import numpy.linalg as nla
import scipy.linalg as sla

from NISQSDP_Ankith import *
from helper_tools import *

Options

In [3]:
optionsQt=qt.Options()
optionsQt.store_final_state=True
optionsQt.store_states=True
optionsQt.nsteps=10000

os.environ["MKL_NUM_THREADS"] = "1"

In [4]:
np.random.seed(seed)

## parameters for fix_overlap, only used when n_params > 1
parameter_list = np.linspace(0, 1, num = n_params)

hilbertspace = 2**n_qubits

expand_elements = np.arange(1, max_states)

## only for POVM, number of random Paulis for expansion
if model == 30:
    ## number of pauli to sample for POVM basis, needs to be larger than max(expand_elements)
    n_pauli_sample = 100
    expand_elements = [6]
    
if do_SDP or model == 30:
    do_SDP = True
    import matlab.engine
    matlab_engine = matlab.engine.start_matlab()
    
qt_dims = [
            [2 for _ in range(n_qubits)],
            [2 for _ in range(n_qubits)]
          ]

# options for solver taken from qutip
opt = qt.Options()

fidelity_list = []
evolved_QAE_state_list = []
E_matrix_list = []
D_matrix_list = []
beta_solution_list = []
prob_meas_POVM_list = []
overlap_states_list = []
qae_result_list = [] ## store alpha or beta of qae

In [5]:
qt_dims

[[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]]

In [6]:
start_time = time.time()

dataset path

In [7]:
dataset = f"NISQSDP__L={n_qubits}__i={ini_type}__m={model}__J={J}__h={h}__d={depth}__R={seed}__M={max_states}__E={max_expansion}"
if ini_type == 15:
    dataset += f"__T={annealtime}"
if model == 30:
    dataset += f"__n={n_POVM}__s={slack_overlap}__f={fix_overlap}"
if n_params > 1:
    dataset += f"__p={n_params}"
dataset += f"__c={np.log10(inv_cond)}"
if get_conserved_number:
    dataset += f"__C={conserved_number}"
dataset = dataset.replace(".","_")
print(f"Dataset = {dataset}:")

# dataset = "NISQSDP"+"L"+str(n_qubits)+"i"+str(ini_type)+"m"+str(model)+"J"+str(J)+"h"+str(h)+"d"+str(depth)+"R"+str(seed)+"M"+str(max_states)+"E"+str(max_expansion)
# if(ini_type==15):
#     dataset+="T"+str(annealtime)
# if(model==30):
#     dataset+="n"+str(n_POVM)+"s"+str(slack_overlap)+"f"+str(fix_overlap)
# if(n_params>1):
#     dataset+="p"+str(n_params)
# dataset+="c"+str(np.log10(inv_cond))
# if(get_conserved_number==True):
#     dataset+="C"+str(conserved_number)

Dataset = NISQSDP__L=6__i=2__m=0__J=1__h=1__d=10__R=1__M=0__E=2__c=-10_0__C=0:


Define operators

In [8]:
opZ = [genFockOp(qt.sigmaz(), i, n_qubits, levels) for i in range(n_qubits)]
opX = [genFockOp(qt.sigmax(), i, n_qubits, levels) for i in range(n_qubits)]
opY = [genFockOp(qt.sigmay(), i, n_qubits, levels) for i in range(n_qubits)]
opId = genFockOp(qt.qeye(levels), 0, n_qubits, levels)

if n_qubits > 1:
    opcsign = [qt.qip.operations.csign(n_qubits, i, (i+1) % n_qubits) for i in range(n_qubits)]

opZ = [$Z \otimes I^{\otimes_5}$, $I \otimes Z \otimes I^{\otimes_4}$, $I^{\otimes_2} \otimes Z \otimes I^{\otimes_3}$, $I^{\otimes_3} \otimes Z \otimes I^{\otimes_2}$, $I^{\otimes_4} \otimes Z \otimes I$, $I^{\otimes_5} \otimes Z$]

In [9]:
opZ

[Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[ 1.  0.  0. ...  0.  0.  0.]
  [ 0.  1.  0. ...  0.  0.  0.]
  [ 0.  0.  1. ...  0.  0.  0.]
  ...
  [ 0.  0.  0. ... -1.  0.  0.]
  [ 0.  0.  0. ...  0. -1.  0.]
  [ 0.  0.  0. ...  0.  0. -1.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[ 1.  0.  0. ...  0.  0.  0.]
  [ 0.  1.  0. ...  0.  0.  0.]
  [ 0.  0.  1. ...  0.  0.  0.]
  ...
  [ 0.  0.  0. ... -1.  0.  0.]
  [ 0.  0.  0. ...  0. -1.  0.]
  [ 0.  0.  0. ...  0.  0. -1.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[ 1.  0.  0. ...  0.  0.  0.]
  [ 0.  1.  0. ...  0.  0.  0.]
  [ 0.  0.  1. ...  0.  0.  0.]
  ...
  [ 0.  0.  0. ... -1.  0.  0.]
  [ 0.  0.  0. ...  0. -1.  0.]
  [ 0.  0.  0. ...  0.  0. -1.]],
 Quantum object: dims = [[2,

opX = [$X \otimes I^{\otimes_5}$, $I \otimes X \otimes I^{\otimes_4}$, $I^{\otimes_2} \otimes X \otimes I^{\otimes_3}$, $I^{\otimes_3} \otimes X \otimes I^{\otimes_2}$, $I^{\otimes_4} \otimes X \otimes I$, $I^{\otimes_5} \otimes X$]

In [10]:
opX

[Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0. 0. 0. 

opY = [$Y \otimes I^{\otimes_5}$, $I \otimes Y \otimes I^{\otimes_4}$, $I^{\otimes_2} \otimes Y \otimes I^{\otimes_3}$, $I^{\otimes_3} \otimes Y \otimes I^{\otimes_2}$, $I^{\otimes_4} \otimes Y \otimes I$, $I^{\otimes_5} \otimes Y$]

In [11]:
opY

[Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  ...
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  ...
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
  [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0.+0.j 0.

opId = $I_{64}$

In [12]:
opId

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
Qobj data =
[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]

Compute target operators

In [13]:
target_op = []
## whether to use sparse solver to get ground state as reference
sparse_gs = n_qubits > 10

# Get Hamiltonian
if model in [0, 1, 14]:
    Hstrings, HpauliFactor, Hvalues, Hoffset = get_Hamiltonian_string(n_qubits, model, J, h, g)
    H = 0
    for i in range(len(Hvalues)):
        H += Hvalues[i] * get_pauli_op(Hstrings[i], opId, opX, opY, opZ)
    H += Hoffset
    target_op = [H]
    groundstate_energy, states_pure = H.groundstate(sparse = sparse_gs)
    print(f"Ground state energy = {groundstate_energy}")
elif model == 30: ## for POVM discrimination get operators
    # ##target_op## plays role of density matrix to be measured here
    # for k in range(n_POVM): # generate states for POVM 
    #     target_dens_matrix = 1/hilbertspace * opId + opX[k]
    #     target_op.append(target_dens_matrix)
    
    ## make sure density matrix is well defined by being trace = 1 and trace(rho^2) <= 1 
    ## (e.g. 1/hilbertspace before opId and )
    target_dens_matrix = 1/hilbertspace * (opId + opX[0])
    target_op.append(target_dens_matrix)
    
    target_dens_matrix = 1/hilbertspace * (opId + opZ[0])
    target_op.append(target_dens_matrix)
    
    Hvalues, H = [], 0
    
n_target_op=len(target_op)

Ground state energy = -3.863703305156273


In [14]:
Hstrings

[[3, 3, 0, 0, 0, 0],
 [0, 3, 3, 0, 0, 0],
 [0, 0, 3, 3, 0, 0],
 [0, 0, 0, 3, 3, 0],
 [0, 0, 0, 0, 3, 3],
 [3, 0, 0, 0, 0, 3],
 [1, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 1]]

In [15]:
HpauliFactor

array([[[0., 0., 0., 1.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 1.],
        [1., 0., 0., 0.]],

       [[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 1.]],

       [[0., 0., 0., 1.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.]],

       [[0., 1., 0., 0.],
        [1., 0., 0., 0.],


In [16]:
Hvalues

[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

In [17]:
Hoffset

0

H = $0.5 * \{Z^{\otimes_2} \otimes I^{\otimes_4} + I \otimes Z^{\otimes_2} \otimes I^{\otimes_3} + I^{\otimes_2} \otimes Z^{\otimes_2} \otimes I^{\otimes_2} + I^{\otimes_3} \otimes Z^{\otimes_2} \otimes I^\otimes + I^{\otimes_4} \otimes Z^{\otimes_2} + X \otimes I^{\otimes_5} + I \otimes X \otimes I^{\otimes_4} + I^{\otimes_2} \otimes X \otimes I^{\otimes_3} + I^{\otimes_3} \otimes X \otimes I^{\otimes_2} + I^{\otimes_4} \otimes X \otimes I + I^{\otimes_5} \otimes X\}$

In [18]:
H

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
Qobj data =
[[3.  0.5 0.5 ... 0.  0.  0. ]
 [0.5 1.  0.  ... 0.  0.  0. ]
 [0.5 0.  1.  ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 1.  0.  0.5]
 [0.  0.  0.  ... 0.  1.  0.5]
 [0.  0.  0.  ... 0.5 0.5 3. ]]

In [19]:
target_op

[Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[3.  0.5 0.5 ... 0.  0.  0. ]
  [0.5 1.  0.  ... 0.  0.  0. ]
  [0.5 0.  1.  ... 0.  0.  0. ]
  ...
  [0.  0.  0.  ... 1.  0.  0.5]
  [0.  0.  0.  ... 0.  1.  0.5]
  [0.  0.  0.  ... 0.5 0.5 3. ]]]

In [20]:
states_pure

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1]], shape = (64, 1), type = ket
Qobj data =
[[ 0.02150153]
 [-0.04919338]
 [-0.04919338]
 [ 0.03905746]
 [-0.04919338]
 [ 0.14793771]
 [ 0.03905746]
 [-0.04647747]
 [-0.04919338]
 [ 0.08303217]
 [ 0.14793771]
 [-0.09429306]
 [ 0.03905746]
 [-0.09429306]
 [-0.04647747]
 [ 0.03905746]
 [-0.04919338]
 [ 0.14793771]
 [ 0.08303217]
 [-0.09429306]
 [ 0.14793771]
 [-0.51384906]
 [-0.09429306]
 [ 0.14793771]
 [ 0.03905746]
 [-0.09429306]
 [-0.09429306]
 [ 0.08303217]
 [-0.04647747]
 [ 0.14793771]
 [ 0.03905746]
 [-0.04919338]
 [-0.04919338]
 [ 0.03905746]
 [ 0.14793771]
 [-0.04647747]
 [ 0.08303217]
 [-0.09429306]
 [-0.09429306]
 [ 0.03905746]
 [ 0.14793771]
 [-0.09429306]
 [-0.51384906]
 [ 0.14793771]
 [-0.09429306]
 [ 0.08303217]
 [ 0.14793771]
 [-0.04919338]
 [ 0.03905746]
 [-0.04647747]
 [-0.09429306]
 [ 0.03905746]
 [-0.09429306]
 [ 0.14793771]
 [ 0.08303217]
 [-0.04919338]
 [-0.04647747]
 [ 0.03905746]
 [ 0.14793771]
 [-0.04919

Get quantum state used for expansion

In [21]:
initial_state = get_ini_state(ini_type, levels, n_qubits, depth, opcsign, annealtime, model, Hvalues, J, H)

In [22]:
initial_state

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1]], shape = (64, 1), type = ket
Qobj data =
[[-0.04403974-0.00486536j]
 [-0.01439275+0.05369134j]
 [ 0.06989705-0.13810728j]
 [-0.06030766-0.04698494j]
 [-0.07516105+0.01927054j]
 [-0.09571324-0.01697809j]
 [ 0.11566225+0.05740516j]
 [-0.01965266-0.021903j  ]
 [ 0.03775345-0.01620539j]
 [ 0.0267553 -0.04728525j]
 [-0.08258505+0.05791781j]
 [ 0.04517539-0.00919324j]
 [ 0.03056897-0.00759581j]
 [ 0.00659133+0.07268039j]
 [-0.03055825-0.11175688j]
 [-0.03554986-0.00763119j]
 [-0.01247543-0.00419684j]
 [ 0.00695517+0.00728736j]
 [-0.00937598-0.06060031j]
 [-0.03161077-0.01504918j]
 [-0.02788917+0.02633365j]
 [-0.05616469+0.03741351j]
 [ 0.05840257-0.02773892j]
 [-0.02019961-0.00612517j]
 [-0.06644026+0.04321626j]
 [-0.02688966+0.08815653j]
 [ 0.0967139 -0.14408001j]
 [-0.08198255+0.03997617j]
 [-0.05412562+0.04231981j]
 [-0.05795641-0.09764391j]
 [ 0.11774441+0.157352j  ]
 [ 0.0577667 +0.00292249j]
 [-0.06556384+0.02071604j]
 [ 0

Calculate ground state energy fror models except POVM

In [23]:
if model == 30:
    pass
else:
    if get_conserved_number:
        # only when there are conserved numbers,
        if model == 1: # excitation number
            H_conserved = sum([opZ[i] for i in range(n_qubits)])
        elif model == 0: # parity
            H_conserved = prod([opX[i] for i in range(n_qubits)])
        
        H_sq_conserved = H_conserved * H_conserved
        n_target_op = 3
        target_op = [H, H_conserved, H_sq_conserved]
        
        ## get ground state of symmetry sector by shifting all eigenvalues in other symmetry 
        ## sectors by a large margin
        conserved_evals, conserved_evecs = nla.eigh(H_conserved.data.toarray())
        modified_evals = np.abs(conserved_evals - conserved_number) * 10**4
        
        # Hamiltonian where unfitting symmetries are shifted by a large margin
        shifted_conserved_Hamiltonian = H + qt.Qobj(conserved_evecs @ np.diag(modified_evals) \
                                                    @ conserved_evecs.conj().T, dims = qt_dims)
        # shifted_conserved_Hamiltonian = H + qt.Qobj(np.dot(conserved_eigenstates, 
        #                                             np.dot(np.diag(modified_eigvals),
        #                                             np.transpose(np.conjugate(conserved_eigenstates)))),
        #                                             dims=qt_dims)
        
        groundstate_energy, states_pure = shifted_conserved_Hamiltonian.groundstate()
        print(f"Ground state energy in symmetry sector = {groundstate_energy}")
    else:
        n_target_op = 1
        get_conserved_number = False

Ground state energy in symmetry sector = 9996.136296694842


H_conserved = $X^{\otimes_6}$

In [24]:
H_conserved

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
Qobj data =
[[0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 ...
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]]

H_sq_conserved = $X^{\otimes_6}X^{\otimes_6} = I_{64}$

In [25]:
H_sq_conserved

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
Qobj data =
[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]

target_op = [$0.5 * \{Z^{\otimes_2} \otimes I^{\otimes_4} + I \otimes Z^{\otimes_2} \otimes I^{\otimes_3} + I^{\otimes_2} \otimes Z^{\otimes_2} \otimes I^{\otimes_2} + I^{\otimes_3} \otimes Z^{\otimes_2} \otimes I^\otimes + I^{\otimes_4} \otimes Z^{\otimes_2} + X \otimes I^{\otimes_5} + I \otimes X \otimes I^{\otimes_4} + I^{\otimes_2} \otimes X \otimes I^{\otimes_3} + I^{\otimes_3} \otimes X \otimes I^{\otimes_2} + I^{\otimes_4} \otimes X \otimes I + I^{\otimes_5} \otimes X\}$, $X^{\otimes_6}$, $I_{64}$]

In [26]:
target_op

[Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[3.  0.5 0.5 ... 0.  0.  0. ]
  [0.5 1.  0.  ... 0.  0.  0. ]
  [0.5 0.  1.  ... 0.  0.  0. ]
  ...
  [0.  0.  0.  ... 1.  0.  0.5]
  [0.  0.  0.  ... 0.  1.  0.5]
  [0.  0.  0.  ... 0.5 0.5 3. ]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[0. 0. 0. ... 0. 0. 1.]
  [0. 0. 0. ... 0. 1. 0.]
  [0. 0. 0. ... 1. 0. 0.]
  ...
  [0. 0. 1. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [1. 0. 0. ... 0. 0. 0.]],
 Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
 Qobj data =
 [[1. 0. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 0. 1. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 1. 0. 0.]
  [0. 0. 0. ... 0. 1. 0.]
  [0. 0. 0. ... 0. 0. 1.]]]

In [27]:
conserved_evals

array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.])

In [28]:
conserved_evecs

array([[-0.70710678+0.j,  0.        +0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j,  0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j, -0.70710678+0.j],
       [ 0.        +0.j, -0.70710678+0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j,  0.        +0.j],
       ...,
       [ 0.        +0.j,  0.70710678+0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j,  0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j, -0.70710678+0.j],
       [ 0.70710678+0.j,  0.        +0.j,  0.        +0.j, ...,
         0.        +0.j,  0.        +0.j,  0.        +0.j]])

In [29]:
modified_evals

array([10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.,
       10000., 10000., 10000., 10000., 10000., 10000., 10000., 10000.])

# WHY DO THIS?

In [30]:
shifted_conserved_Hamiltonian

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2]], shape = (64, 64), type = oper, isherm = True
Qobj data =
[[1.0003e+04 5.0000e-01 5.0000e-01 ... 0.0000e+00 0.0000e+00 0.0000e+00]
 [5.0000e-01 1.0001e+04 0.0000e+00 ... 0.0000e+00 0.0000e+00 0.0000e+00]
 [5.0000e-01 0.0000e+00 1.0001e+04 ... 0.0000e+00 0.0000e+00 0.0000e+00]
 ...
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 1.0001e+04 0.0000e+00 5.0000e-01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 0.0000e+00 1.0001e+04 5.0000e-01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 5.0000e-01 5.0000e-01 1.0003e+04]]

In [31]:
states_pure

Quantum object: dims = [[2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1]], shape = (64, 1), type = ket
Qobj data =
[[-0.02150153]
 [ 0.04919338]
 [ 0.04919338]
 [-0.03905746]
 [ 0.04919338]
 [-0.14793771]
 [-0.03905746]
 [ 0.04647747]
 [ 0.04919338]
 [-0.08303217]
 [-0.14793771]
 [ 0.09429306]
 [-0.03905746]
 [ 0.09429306]
 [ 0.04647747]
 [-0.03905746]
 [ 0.04919338]
 [-0.14793771]
 [-0.08303217]
 [ 0.09429306]
 [-0.14793771]
 [ 0.51384906]
 [ 0.09429306]
 [-0.14793771]
 [-0.03905746]
 [ 0.09429306]
 [ 0.09429306]
 [-0.08303217]
 [ 0.04647747]
 [-0.14793771]
 [-0.03905746]
 [ 0.04919338]
 [ 0.04919338]
 [-0.03905746]
 [-0.14793771]
 [ 0.04647747]
 [-0.08303217]
 [ 0.09429306]
 [ 0.09429306]
 [-0.03905746]
 [-0.14793771]
 [ 0.09429306]
 [ 0.51384906]
 [-0.14793771]
 [ 0.09429306]
 [-0.08303217]
 [-0.14793771]
 [ 0.04919338]
 [-0.03905746]
 [ 0.04647747]
 [ 0.09429306]
 [-0.03905746]
 [ 0.09429306]
 [-0.14793771]
 [-0.08303217]
 [ 0.04919338]
 [ 0.04647747]
 [-0.03905746]
 [-0.14793771]
 [ 0.04919

Generate all possible Pauli strings

In [32]:
n_paulis = 2**n_qubits
all_pauli_list = np.zeros([n_paulis, n_qubits], dtype = int)
for k in range(n_paulis):
    all_pauli_list[k, :] = numberToBase(k, 2, n_qubits)

In [33]:
all_pauli_list

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 0, 1],
       [0, 0, 0, 1, 1, 0],
       [0, 0, 0, 1, 1, 1],
       [0, 0, 1, 0, 0, 0],
       [0, 0, 1, 0, 0, 1],
       [0, 0, 1, 0, 1, 0],
       [0, 0, 1, 0, 1, 1],
       [0, 0, 1, 1, 0, 0],
       [0, 0, 1, 1, 0, 1],
       [0, 0, 1, 1, 1, 0],
       [0, 0, 1, 1, 1, 1],
       [0, 1, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 1],
       [0, 1, 0, 0, 1, 0],
       [0, 1, 0, 0, 1, 1],
       [0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 1],
       [0, 1, 0, 1, 1, 0],
       [0, 1, 0, 1, 1, 1],
       [0, 1, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 1],
       [0, 1, 1, 0, 1, 0],
       [0, 1, 1, 0, 1, 1],
       [0, 1, 1, 1, 0, 0],
       [0, 1, 1, 1, 0, 1],
       [0, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1],
       [1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 1, 0],
       [1, 0, 0, 0, 1, 1],
       [1, 0, 0, 1, 0, 0],
 

Generate Paulis for state spaces used for finding POVMs

In [34]:
if model == 30:
    ## randomly sample various pauli operators for expansion of state

    ## all zeros is added first to make ensure base state \psi is part of expansion
    all_pauli_list = np.concatenate((np.zeros([1, n_qubits], dtype = int),
                                     np.random.randint(0, 4, size=[n_pauli_sample-1, n_qubits])))
    all_pauli_list = np.unique(all_pauli_list, axis = 0) ## only take unique paulis for expansion
    np.random.shuffle(all_pauli_list[1:]) ## shuffle all except identity pauli

Generate K-moment expansion that serves as linear combination basis of the subspace

# DID NOT UNDERSTAND!

In [35]:
base_expand_strings = [np.zeros(n_qubits, dtype = int)]
if model in [0, 1, 14]:
    for k in range(max_expansion):
        # do moment expansion with Hamiltonian terms.
        base_expand_strings += list(multiply_paulis(base_expand_strings, Hstrings)[0])
        new_strings, string_index = list(np.unique(base_expand_strings, axis = 0, return_index = True))
        sorted_index = np.sort(string_index)
        base_expand_strings = [base_expand_strings[k] for k in sorted_index]
        # base_expand_strings = list(new_strings)
        if max_states > 0:
            base_expand_strings = base_expand_strings[:max_states]
    # split = 2 * L + 1
    # split_base = list(base_expand_strings[:split])
    # base_expand_strings = base_expand_strings[split:]
    # np.random.shuffle(base_expand_strings)
    # base_expand_strings = np.array(split_base + list(base_expand_strings))
    
    # if max_states > 0:
    #     base_expand_strings = base_expand_strings[:max_states]
    
    all_pauli_list = base_expand_strings
    
    if not max_states:
        expand_elements = np.unique(list(np.arange(1, len(all_pauli_list) + 1, 2)) + [len(all_pauli_list)])
        
expand_order = len(expand_elements)

In [36]:
all_pauli_list

[array([0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 1]),
 array([0, 0, 0, 0, 1, 0]),
 array([0, 0, 0, 0, 3, 3]),
 array([0, 0, 0, 1, 0, 0]),
 array([0, 0, 0, 3, 3, 0]),
 array([0, 0, 1, 0, 0, 0]),
 array([0, 0, 3, 3, 0, 0]),
 array([0, 1, 0, 0, 0, 0]),
 array([0, 3, 3, 0, 0, 0]),
 array([1, 0, 0, 0, 0, 0]),
 array([3, 0, 0, 0, 0, 3]),
 array([3, 3, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 1, 1]),
 array([0, 0, 0, 0, 2, 3]),
 array([0, 0, 0, 0, 3, 2]),
 array([0, 0, 0, 1, 0, 1]),
 array([0, 0, 0, 1, 1, 0]),
 array([0, 0, 0, 1, 3, 3]),
 array([0, 0, 0, 2, 3, 0]),
 array([0, 0, 0, 3, 0, 3]),
 array([0, 0, 0, 3, 2, 0]),
 array([0, 0, 0, 3, 3, 1]),
 array([0, 0, 1, 0, 0, 1]),
 array([0, 0, 1, 0, 1, 0]),
 array([0, 0, 1, 0, 3, 3]),
 array([0, 0, 1, 1, 0, 0]),
 array([0, 0, 1, 3, 3, 0]),
 array([0, 0, 2, 3, 0, 0]),
 array([0, 0, 3, 0, 3, 0]),
 array([0, 0, 3, 2, 0, 0]),
 array([0, 0, 3, 3, 0, 1]),
 array([0, 0, 3, 3, 1, 0]),
 array([0, 0, 3, 3, 3, 3]),
 array([0, 1, 0, 0, 0, 1]),
 array([0, 1, 0, 0, 

In [37]:
expand_elements

array([ 1,  3,  5,  7,  9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33,
       35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67,
       69, 71, 73, 75, 77, 79])

In [38]:
expand_order

40

Expand to expand_order moments

In [39]:
max(expand_elements)

79

In [15]:
# this loop is to vary overlap for POVM, does only one iteration for hamiltonian stuff
for pp in range(len(parameter_list)):
    if len(parameter_list) > 1:
        fix_overlap = parameter_list[pp]
        
    #get maximal subspace 
    expand_strings = all_pauli_list[:max(expand_elements)]
    expand_states = []
    n_expand_states = max(expand_elements)
    for i in range(n_expand_states):
        expand_states.append(get_pauli_op(expand_strings[i], opId, opX, opY, opZ) * initial_state)
    
    E_matrix_all = np.zeros([n_expand_states, n_expand_states], dtype = np.complex128)
    D_matrix_all = np.zeros([n_target_op, n_expand_states, n_expand_states], dtype = np.complex128)
    
    for m in range(n_expand_states):
        for n in range(n_expand_states):
            E_matrix_all[m,n] = expand_states[m].overlap(expand_states[n])
            for k in range(n_target_op):
                D_matrix_all[k][m,n] = (expand_states[m].overlap(target_op[k] * expand_states[n]))
        
    ## now go through subspace for a smaller amount of basis states to see convergence 
    ## with increasing number of subspace states
    for p_counter, p in enumerate(expand_elements):
        expand_strings = all_pauli_list[:p]
        n_expand_states = len(expand_strings)
        print(f"\nOrder {p}: n states = {n_expand_states}")
        E_matrix = E_matrix_all[:n_expand_states, :][:, :n_expand_states]
        D_matrix = [D_matrix_all[k][:n_expand_states, :][:, :n_expand_states] for k in range(n_target_op)]
        E_matrix_list.append(E_matrix)
        D_matrix_list.append(D_matrix)
    
        # adjust E_matrix to make it numerical stable for positive definite
        
        # E_matrix can be not positive definite due to numerical issues. 
        # Diagonalize matrix, and set negative evals to positive value
        # this is only needed for QAE, not for QAS
        
        e_vals, e_vecs = sla.eigh(E_matrix)
    #    e_vals_adjusted = np.array(e_vals)
    #    for k in range(len(e_vals_adjusted)):
    #        if e_vals_adjusted[k] < epsilon:
    #            e_vals_adjusted[k] = epsilon
    #    E_matrix_corrected = np.dot(e_vecs, np.dot(np.diag(e_vals_adjusted), np.transpose(np.conjugate(e_vecs))))
    #    
    #        
        ## calculate inverse of E
        ## inv_cond determines threshold for SVD values: below inv_cond, value is set to zero. 
        ## This is to avoid problems with small eigenvalues, which can blow up with inversion
        ## E_inv = np.linalg.pinv(E_matrix, hermitian = True, rcond = inv_cond)
        
        # choose initial alpha. Here, we choose initial_state as beginning state. 
        # In principle, any alpha can be chosen. One could also use QAE to find some state
        ini_alpha = np.zeros(n_expand_states, dtype = np.complex128)
    
        # get closest projection of initial evolution state via QAE
        ini_matrix = D_matrix
        
        print("Start SDP")
        e_vals_cond = np.array(e_vals)
        for k in range(len(e_vals_cond)):
            if e_vals_cond[k] < inv_cond:
                e_vals_cond[k] = 0
                
        E_matrix_corrected = e_vecs @ np.diag(e_vals_cond) @ e_vecs.conj().T
        # E_matrix_corrected = np.dot(e_vecs,np.dot(np.diag(e_vals_cond),np.transpose(np.conjugate(e_vecs))))
        E_matrix_cp = E_matrix # E_matrix_corrected
        
        if model == 30: # POVM
            curent_dir = os.getcwd()
            matlab_engine.cd(curent_dir)
            
            ## get pure states 
            states_vector = [2 * np.random.rand(n_expand_states) - 1 for _ in range(n_POVM)]
            
            ## normalise state
            for k in range(n_POVM):
                states_vector[k] = states_vector[k] / np.sqrt(states_vector[k].conj() \
                                                              @ E_matrix_cp @ states_vector[k])
                # states_vector[k] = states_vector[k] / np.sqrt(np.dot(np.conjugate(states_vector[k]),np.dot(E_matrix_cp,states_vector[k])))
            
            if fix_overlap >= 0: ## adjust overlap between states to be discriminated
                if n_POVM == 2:
                    ## get orthogonal states
                    states_vector[1] -= (states_vector[0] @ E_matrix_cp @ states_vector[1]) * states_vector[0]
                    states_vector[1] /= np.sqrt(states_vector[1].conj() @ E_matrix_cp @ states_vector[1])
                    
                    # states_vector[1] = states_vector[1]-np.dot(np.conjugate(states_vector[0]),np.dot(E_matrix_cp,states_vector[1]))*states_vector[0]
                    # states_vector[1]=states_vector[1]/np.sqrt(np.dot(np.conjugate(states_vector[1]),np.dot(E_matrix_cp,states_vector[1])))
                
                # interpolate overlaps
                states_vector[1] = (1 - fix_overlap) * states_vector[1] \
                                    + fix_overlap * states_vector[0]
                states_vector[1] /= np.sqrt(states_vector[1].conj() @ E_matrix_cp @ states_vector[1])
                # states_vector[1] = states_vector[1]/np.sqrt(np.dot(np.conjugate(states_vector[1]),np.dot(E_matrix_cp,states_vector[1])))
                
            ## construct matrix description
            states_matrix = [np.outer(states_vector[k].conj(), states_vector[k]) \
                             for k in range(n_POVM)]
            overlap_states = [[np.trace(states_matrix[q] @ E_matrix_cp @ states_matrix[k] @ E_matrix_cp) for k in range(n_POVM)] for q in range(n_POVM)]
            # states_matrix = [np.outer(np.conjugate(states_vector[k]),states_vector[k]) for k in range(n_POVM)]
            # overlap_states=[[np.trace(np.dot(states_matrix[q],np.dot(E_matrix_cp,np.dot(states_matrix[k],E_matrix_cp)))) for k in range(n_POVM)] for q in range(n_POVM)]
            
            if n_expand_states > 1: # run SDP
                slack_overlap = float(slack_overlap)
                mat_E_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, "like", 1j)
                #mat_D_matrix = matlab_engine.zeros(n_POVM, n_expand_states, n_expand_states, "like", 1j)
                mat_states_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, n_POVM, "like", 1j)
    
                for i in range(n_expand_states):
                    for j in range(n_expand_states):
                        mat_E_matrix[i][j] = E_matrix_cp[i, j]
                        for k in range(n_POVM):
                            # mat_D_matrix[k][i][j]=D_matrix[k,i,j]
                            mat_states_matrix[i][j][k] = states_matrix[k][i,j]
                        
                # a = matlab_engine.matlabSDP(1, nargout = 1)
                print("Run Matlab")
                #beta_solution_matlab = matlab_engine.matlabPOVMSDP(mat_E_matrix,mat_D_matrix,slack_overlap)
                beta_solution_matlab = matlab_engine.matlabPOVM_EMatrix_SDP(mat_E_matrix, \
                                                                            mat_states_matrix, \
                                                                            slack_overlap)
                print("Finished Matlab")
                beta_solution = np.array(beta_solution_matlab)
                beta_solution = np.swapaxes(beta_solution, 0, 2)
                beta_solution = np.swapaxes(beta_solution, 1, 2)
                
                # prob_correct_meas = [np.trace(np.dot(beta_solution[:,:,k],np.dot(mat_E_matrix,np.dot(mat_states[k],mat_E_matrix)))) for k in range(n_POVM)]
                # prob_correct_meas = [np.trace(beta_solution[:, :, k] @ mat_E_matrix \
                #                               @ mat_states_matrix[k] @ mat_E_matrix) \
                #                      for k in range(n_POVM)]
    
                if np.isnan(beta_solution).any():
                    print("WARN: Solution has nan, replace with default")
                    beta_solution = np.zeros([n_POVM, n_expand_states, \
                                              n_expand_states], dtype = np.complex128)
    
                ## first index is state, second index is POVM
                prob_meas_POVM = [[np.real(np.trace(beta_solution[k, :, :] @ E_matrix_cp @ states_matrix[q] @ E_matrix_cp)) for k in range(n_POVM)] for q in range(n_POVM)]
                # prob_meas_POVM = [[np.real(np.trace(np.dot(beta_solution[k,:,:],np.dot(E_matrix_cp,np.dot(states_matrix[q],E_matrix_cp))))) for k in range(n_POVM)] for q in range(n_POVM)]
            else:
                beta_solution = np.array([np.array([[1]]) for k in range(n_POVM)])
                prob_meas_POVM = [[1]]
                
            # sum over probabilities of POVM to find right states
            obj_val = np.sum(np.diag(prob_meas_POVM))
            meas_error = np.sum(prob_meas_POVM) - obj_val
            print(f"Meas correct = {obj_val}, Meas wrong = {meas_error}")
            
            if n_POVM == 2:
                ## exact theory for 2 pure states and unambigious discrimination (slack_overlap = 0)
                theta = np.arccos(np.sqrt(np.real(overlap_states[0][1]))) ## angle between two states
                theory_obj = 2 * (1 - np.cos(theta))
                print(f"Theory = {theory_obj}, actual = {obj_val}")
            
            beta_solution_list.append(beta_solution)
            prob_meas_POVM_list.append(prob_meas_POVM)
            overlap_states_list.append(overlap_states)
            
            beta_qae = beta_solution
    
        if model in [0, 1, 14]: # find Hamiltonian ground state
            ini_matrix = D_matrix[0]
            if do_SDP: # do SDP to find Hamiltonian ground state
                shift_value = -10 ## shift lowest soluton to negative so we do not get zero solution by minimization
                
                cur_dir = os.getcwd()
                matlab_engine.cd(cur_dir)
                
                if n_expand_states > 1:
                    mat_E_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, "like", 1j)
                    mat_D_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, "like", 1j)
                    mat_C_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, "like", 1j)
                    mat_C2_matrix = matlab_engine.zeros(n_expand_states, n_expand_states, "like", 1j)
                    
                    if get_conserved_number:
                        for i in range(n_expand_states):
                            for j in range(n_expand_states):
                                mat_C_matrix[i][j] = D_matrix[1][i,j]
                                mat_C2_matrix[i][j] = D_matrix[2][i,j]
                        
                    for i in range(n_expand_states):
                        for j in range(n_expand_states):
                            mat_E_matrix[i][j] = E_matrix_cp[i,j]
                            mat_D_matrix[i][j] = ini_matrix[i,j]
                            
                    # a = matlab_engine.matlabSDP(1, nargout = 1)
                    print("Run Matlab")
                    beta_solution_matlab = matlab_engine.matlabSDP(mat_E_matrix, mat_D_matrix, 
                                                                   float(shift_value), \
                                                                   get_conserved_number, \
                                                                   float(conserved_number), \
                                                                   mat_C_matrix, mat_C2_matrix)
                    print("Finished Matlab")
                    beta_solution = np.array(beta_solution_matlab)
                    
                    if np.isnan(beta_solution).any():
                        print("WARN: Solution has nan, replace with default")
                        beta_solution = np.zeros([n_expand_states, n_expand_states], \
                                                 dtype = np.complex128)
                        beta_solution[0,0] = 1
                else:
                    beta_solution = np.array([[1]])
                
                # Obj fn
                val_soln = np.real(np.trace(beta_solution @ ini_matrix)) ## remove the shift
                # Constraints
                trace_E = np.real(np.trace(beta_solution @ E_matrix_cp))
                purity_E = np.real(np.trace(beta_solution @ E_matrix_cp @ beta_solution @ E_matrix_cp))
                # purity_E = np.real(np.trace(np.dot(np.dot(beta_solution,E_matrix_cp),np.dot(beta_solution,E_matrix_cp))))
                
                print(f"<Beta, E> = {trace_E}, purity = {purity_E}")
                print(f"SDP value is {val_soln}")
                
                beta_solution_list.append(beta_solution)
                beta_qae = beta_solution
                
                # eigval, eigvec = nla.eigh(beta_solution)
                
                # print(f"Beta eigval = {eigval}")
                # ini_alpha_vec = eigvec[:,-1] # take largest eigensvalue of beta
                
                U_matrix = np.hstack([expand_states[i].data.toarray() for i in range(n_expand_states)])
                
                ini_reconstructed_density_matrix = qt.Qobj(U_matrix @ beta_qae @ U_matrix.conj().T, dims = qt_dims)
                # ini_reconstructed_density_matrix = qt.Qobj(np.dot(np.dot(U_matrix, beta_qae), \
                #                                            np.transpose(np.conjugate(U_matrix))),\
                #                                            dims = qt_dims)
                    
                dm_evals, dm_evecs = ini_reconstructed_density_matrix.eigenstates()
                
                ## largest eigenstate of density matrix is reconstructed state
                ini_reconstructed_state = dm_evecs[-1]                
            else: # do generalized eigenvalue problem           
                # get e_matrix evals inverted, cutoff with inv_cond
                e_vals_inverted = np.array(e_vals)
            
                for k in range(len(e_vals_inverted)):
                    e_vals_inverted[k] = 0 if e_vals_inverted[k] < inv_cond else 1/e_vals_inverted[k]

                # calculate ground state via generalized eigenvalue problem D\alpha = \lambda E\alpha
                # qae_energy, qae_vectors = sla.eigh(ini_matrix, E_matrix_corrected)
                
                ### calculate eigenvalues for E^-1 * H\alpha = \lambda \alpha. 
                ### Doesnt seem to work for some resason though ....
                ## E_inv = nla.pinv(E_matrix, hermitian = True, rcond = inv_cond)
                ## qae_energy, qae_vectors = sla.eigh(E_inv @ ini_matrix)
            
                # E_inv = nla.pinv(E_matrix, hermitian = True, rcond = inv_cond)
            
                # convert generalized eigenvalue problem with a regular eigenvalue problem using 
                # "EIGENVALUE PROBLEMS IN STRUCTURAL MECHANICS"
                # we want to solve D\alpha = \lambda E\alpha
                # turns out this does not work well if E_matrix has near zero eigenvalues
                # instead, we turn this into regular eigenvalue problem which is more behaved
                # we diagonalize E_matrix = U * F * F * U^\dag with diagonal F
                # Then, define S = U * F, and S^-1 = F^-1 * U^\dag. 
                # Use conditioned eigenvalues F for this such that no negative eigenvalues appear, 
                # and for inverse large eigenvalues set to zero
                # solve S^-1 * D * S^-1^\dag * a = \lambda a
                # convert \alpha = S^-1^\dag * a. This is the solution to original problem.
                # This procedure ensures that converted eigenvalue problem remains hermitian, 
                # and no other funny business happens
                s_matrix = e_vecs @ np.diag(np.sqrt(e_vals_cond))
                s_matrix_inv = np.diag(np.sqrt(e_vals_inverted)) @ e_vecs.conj().T
                toeigmat = s_matrix_inv @ ini_matrix @ s_matrix_inv.conj().T 
                # toeigmat = np.dot(s_matrix_inv,np.dot(ini_matrix,np.transpose(np.conjugate(s_matrix_inv))))
            
                qae_energy, qae_vectors = sla.eigh(toeigmat)
                # print(qae_energy)
                
                for k in range(len(e_vals_cond)): # go through eigenvectors, take the lowest one that has non-zero norm
                    ini_alpha_vec = qae_vectors[:,k]
                    ini_alpha_vec = s_matrix_inv.conj().T @ ini_alpha_vec
                    # ini_alpha_vec = np.dot(np.transpose(np.conjugate(s_matrix_inv)),ini_alpha_vec)
                
                    norm_ini_alpha = np.sqrt(np.abs(ini_alpha_vec.conj().T @ E_matrix @ ini_alpha_vec))
                    # norm_ini_alpha = np.sqrt(np.abs(np.dot(np.transpose(np.conjugate(ini_alpha_vec)),np.dot(E_matrix,ini_alpha_vec))))
                    if norm_ini_alpha != 0:
                        break
                    
                print(f"norm = {norm_ini_alpha}")
                ini_alpha_vec /= norm_ini_alpha
                        
                qae_result_list.append(ini_alpha_vec)
                
                ini_reconstructed_state = sum([expand_states[i] * ini_alpha_vec[i] for i in range(n_expand_states)])

        # get Fidelity for finding ground state
        if model == 30:
            pass
        else:
            fidIni = np.abs(ini_reconstructed_state.overlap(states_pure)) **2
            fidelity_list.append(fidIni)
            evolved_QAE_state_list.append(ini_reconstructed_state)
            print(f"Initial state fidelity = {fidIni}")


Order 6: n states = 6
Start SDP
Run Matlab
Finished Matlab
Meas correct = 1.742191859887459, Meas wrong = 1.1465579907721235e-07
Theory = 1.7412989224440556, actual = 1.742191859887459


Get expectation values

In [16]:
if model == 30: # for POVM no expectation values 
    expectZ = []
    expectX = []
    expectH = []
    expectD = []
    expectZpure = []
    expectXpure = []
    expectHpure = []
    expectDpure = []
    norm_list = []
else:
    # if noisy_ini_state == 0:        
    # get various expectation values
    expectZ = [[np.real(qt.expect(opZ[k], evolved_QAE_state_list[i])) \
                for k in range(n_qubits)] for i in range(expand_order)]
    
    expectX = [[np.real(qt.expect(opX[k], evolved_QAE_state_list[i])) \
                for k in range(n_qubits)] for i in range(expand_order)]
                
    # energy for static hamiltonian
    expectH = [np.real(qt.expect(target_op[0], evolved_QAE_state_list[i])) \
               for i in range(expand_order)]
    
    # energy for static hamiltonian
    expectD = [[np.real(qt.expect(target_op[k], evolved_QAE_state_list[i])) \
                for k in range(n_target_op)] for i in range(expand_order)]
    
    expectZpure = [np.real(qt.expect(opZ[k], states_pure)) for k in range(n_qubits)]
    expectXpure = [np.real(qt.expect(opX[k], states_pure))  for k in range(n_qubits)]
    expectHpure = np.real(qt.expect(target_op[0], states_pure))
    expectDpure=[np.real(qt.expect(target_op[k], states_pure)) for k in range(n_target_op)] 
    
    ##
    # gibbs = (-H * times[10] * 2).expm()
    # gibbs /= gibbs.tr()
    # qt.fidelity(states_pure[10], gibbs)
    
    legendOrder = []
    xlabelstring = "$M$"
    
    # legendOrder = [f"$K={i}$" for i in range(0, expand_order+1)]
    
    # fidelity with exact evolution
    plot1D([fidelity_list], [expand_elements], saveto = saveto, dataname = "fidelity" + dataset,
           xlabelstring = xlabelstring, ylabelstring = "$F$", marker = False, legend = legendOrder)
    
    
    # norm of evolved state. Should remain one. If not, try increasing inv_cond 
    # norm_list = [evolved_QAE_state_list[i].norm() for i in range(expand_order + 1)]
    # plot1D([norm_list], [expand_elements], saveto = saveto, dataname = "norm" + dataset,
    #        xlabelstring = xlabelstring, ylabelstring = "norm", marker = False)
    norm_list = []
    
    xrange = np.arange(n_qubits)

    # static energy over time
    customMarkerStyle = ["o"] * 2
    customMarkerStyle[-1] = ""
    
    
    legendOrder = ["NISQ SDP", "exact"]
    plot1D([expectH] + [[expectHpure] * (expand_order)], [expand_elements] * 2, saveto = saveto, 
           dataname = "Hpure" + dataset, xlabelstring = xlabelstring, ylabelstring = "$\\langle H\\rangle$",
           legend = legendOrder, customMarkerStyle = customMarkerStyle)
        
    plot1D(np.transpose(expectD), [expand_elements] * n_target_op, saveto = saveto, 
           dataname = "D" + dataset, xlabelstring = xlabelstring, 
           ylabelstring = "$\\langle D\\rangle$")
    
    plot1D([[np.sum([np.abs(expectD[i][k] - expectDpure[k]) for k in range(0,n_target_op)]) \
             for i in range(expand_order)]], [expand_elements], saveto = saveto, 
           dataname = "Ddiff" + dataset, xlabelstring = xlabelstring, 
           ylabelstring = "$\\vert\\langle C\\rangle-C_\mathrm{exact}\\vert$")
        
    # print(f"Energy NISQ SDP = {expectH}, groundstate = {expectHpure}")

In [17]:
print(f"Total time taken = {time.time() - start_time}s")
print(dataset)

Total time taken = 3.1944310665130615s
NISQSDP__L=6__i=2__m=30__J=1__h=1__d=10__R=1__M=0__E=2__n=2__s=0__f=-1__c=-10_0__C=0
